# 42 새 질문 48개로 검색 방식 확대 평가
운영 코드·OCR·이전 결과는 읽기 전용입니다. 실험은 순차 실행합니다.
이 노트북은 입력 → 실행 상태 → 평가 → 비용을 저장 결과로 확인합니다. 유료 호출은 자동 실행하지 않습니다.
장시간 실행·원응답 journaling·동일 payload 캐시·메모리 중단/재개가 필요해 `42_expanded_query_evaluation.py`를 별도 runner로 사용합니다.
재현: 기존 승인 환경에서 `python -B notebooks/scripts/42_expanded_query_evaluation.py` 실행 후 이 노트북의 전체 표시 셀을 실행합니다.
모델·자료·평가 기준은 protocol.json과 소스 해시로 고정합니다. 같은 질문 재호출 분산은 측정하지 않습니다.

완료: 검색384/384,답변383/384. 출력 한도 실패2회는 원래 재시도로 복구됐고,중복 카드 추천1조건은 최종 실패0점입니다. 자동 최고와 원문 대조 후 선택을 구분합니다. 이번 실행은 새 질문 수를 늘린 평가이며 같은 질문 반복 생성의 분산은 미측정입니다.


In [1]:
from pathlib import Path
import json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "notebooks/data/42_expanded_query_evaluation"
def read(name): return json.loads((DATA / name).read_text())
protocol = read("protocol.json")
print(json.dumps(protocol, ensure_ascii=False, indent=2)[:10000])


{
  "diagnostic_behavior": "all4 expected_behavior rules source-reviewed for every configuration; no ungrounded definite benefit/optimality claims; no guard regression allowed",
  "expected_rows": 384,
  "independent_holdout": false,
  "modes": [
    "U",
    "S_THEN_U",
    "S_PLUS_U",
    "Q_EXPAND_U"
  ],
  "numeric_normalization_and_alias_rubric_changes": 0,
  "positive_questions": 44,
  "primary": "known_positive_complete; positive44 denominator; failures0",
  "production_adopted": false,
  "profiles": [
    "card_page_section_benefit",
    "parent_child_bundle"
  ],
  "questions": 48,
  "same_query_repeated_generations": 0,
  "sampling": "16 numeric,16 named-card detail across10 issuers,12 semantic positive,4 advice/ambiguous/source-unconfirmed diagnostics; source-reviewed partial anchors authored before evaluated retrieval/answers",
  "scorer_sha256": "b8b60735c2081a87b96567f4ce9f17d5a18abef0a93b044ff09dcfa107aa9d02",
  "selection": "Q_EXPAND_U vs same-profile U wins>losses plus

## 실행 결과
검색·답변 실패도 분모에 포함합니다. 일반/모호/자료 미확인 질문은 양성 지표에서 분리합니다.


In [2]:
print(json.dumps(read("run_summary.json"), ensure_ascii=False, indent=2))


{
  "answer_rows": 384,
  "answer_success_rows": 383,
  "cost": {
    "billed_cost_usd": null,
    "duplicate_saved_responses": 0,
    "known_cost_usd": "0.319535615",
    "known_priced_records": 307,
    "record_count": 307,
    "tokens": {
      "cache_write_tokens": 1254855,
      "cached_input_tokens": 691426,
      "embedding_tokens": 3364,
      "input_tokens": 1947193,
      "ordinary_input_tokens": 912,
      "output_tokens": 311212
    },
    "unknown_cost_records": 0
  },
  "expected": 384,
  "production_adopted": false,
  "resources": {
    "billed_cost": "미측정",
    "cpu_seconds_this_process": 16825.532502,
    "gpu": "CPU 실행",
    "last_available_kib": 2654448,
    "last_required_kib": 1048576,
    "minimum_available_kib": 2508804,
    "new_bge_pairs_this_process": 2925,
    "peak_rss_kib": 2778396,
    "singleton_calls": 2925,
    "transmitted_tokens": "미측정",
    "wall_seconds_this_process": 7537.0986236430035
  },
  "retrieval_rows": 384,
  "retrieval_success_rows": 384
}

## 평가
Hit@3는 상위3개 정답 카드 포함, MRR은 첫 정답 순위 역수 평균, 답변충족은 원문 상대 자동 조건 검사의 전체 통과입니다. 높을수록 좋습니다.
카드가 검색돼도 혜택·조건을 답변에서 빠뜨리면 답변충족에 실패할 수 있습니다. 문자열·동의어·부정문 오탐과 불완전한 원문 관계를 원응답으로 따로 점검합니다.
benefit과 parent-child는 청킹 외 재정렬 프로필에도 차이가 있습니다.


In [3]:
import pandas as pd
evaluation = read("evaluation_summary.json")
display(pd.DataFrame(evaluation["metrics"]))
print(json.dumps(evaluation["paired"], ensure_ascii=False, indent=2))

print(json.dumps(read("decision.json"), ensure_ascii=False, indent=2))
review = read("quality_review.json")
print("원문 점검 조건:", review["inspected_conditions"], "필수 조건:", review["required_conditions"], "동결 점수 변경:", review["scores_changed"])


,all_n,complete,completion_wilson_nominal95,failures,group,hit,mode,mrr,ownership_failures,positive_n,profile,scope_failures,unsupported_percent_claims
0,48,30,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,30,U,0.579545,0,44,card_page_section_benefit,0,0
1,48,31,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,35,S_THEN_U,0.685606,0,44,card_page_section_benefit,0,1
2,48,33,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,35,S_PLUS_U,0.670455,0,44,card_page_section_benefit,0,1
3,48,31,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,32,Q_EXPAND_U,0.632576,0,44,card_page_section_benefit,0,1
4,48,26,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,32,U,0.590909,0,44,parent_child_bundle,0,0
5,48,30,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,34,S_THEN_U,0.655303,0,44,parent_child_bundle,0,0
6,48,30,"{'confidence': 0.95, 'limitation': 'Nominal bi...",1,all,34,S_PLUS_U,0.643939,1,44,parent_child_bundle,1,1
7,48,26,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,all,31,Q_EXPAND_U,0.602273,0,44,parent_child_bundle,0,0
8,16,12,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,numeric,12,U,0.510417,0,16,card_page_section_benefit,0,0
9,16,12,"{'confidence': 0.95, 'limitation': 'Nominal bi...",0,numeric,14,S_THEN_U,0.677083,0,16,card_page_section_benefit,0,1


[
  {
    "baseline": "U",
    "candidate": "Q_EXPAND_U",
    "changed": [
      {
        "query_id": "E02",
        "question": "CU 결제액을 7% 이상 적립해주는 카드가 있을까",
        "result": "win"
      },
      {
        "query_id": "E31",
        "question": "카드의정석2의 국내외 가맹점 할인 혜택과 적용 조건을 근거로 설명해줘",
        "result": "loss"
      },
      {
        "query_id": "E44",
        "question": "대형마트에서 장볼 때 받을 수 있는 카드 혜택",
        "result": "win"
      }
    ],
    "losses": 1,
    "profile": "card_page_section_benefit",
    "ties": 41,
    "wins": 2
  },
  {
    "baseline": "S_THEN_U",
    "candidate": "Q_EXPAND_U",
    "changed": [
      {
        "query_id": "E05",
        "question": "네이버쇼핑에서 5% 이상 할인받고 싶어",
        "result": "loss"
      },
      {
        "query_id": "E14",
        "question": "GS25와 넷플릭스를 각각 10% 이상 할인해주는 카드 한 장",
        "result": "win"
      },
      {
        "query_id": "E15",
        "question": "병원과 약국에서 모두 5% 이상 할인되는 카드를 찾아줘",
        "result": "win"
      },
      {
      

## 비용·원본 보존
API 실제 물리 요청과 캐시 재사용을 분리합니다. 사용량×기록 단가 환산이며 실제 청구액은 미측정입니다. 시간·CPU·peak RSS는 해당 프로세스의 측정값입니다.


In [4]:
print(json.dumps(read("cost_accounting.json")["summary"], ensure_ascii=False, indent=2))
print(json.dumps(read("preservation.json"), ensure_ascii=False, indent=2))


{
  "billed_cost_usd": null,
  "duplicate_saved_responses": 0,
  "known_cost_usd": "0.319535615",
  "known_priced_records": 307,
  "record_count": 307,
  "tokens": {
    "cache_write_tokens": 1254855,
    "cached_input_tokens": 691426,
    "embedding_tokens": 3364,
    "input_tokens": 1947193,
    "ordinary_input_tokens": 912,
    "output_tokens": 311212
  },
  "unknown_cost_records": 0
}
{
  "changed": [],
  "checked_files": 20631,
  "passed": true
}
